In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [5]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    n_lags = 1 #hardcoded to 1 for now, can be reintroduced in the future
    # Search space
    window_size = trial.suggest_int("window_size", 10, 400, step=10)
    #n_lags      = trial.suggest_int("n_lags", 1, 25)
    lam         = trial.suggest_float("lambda", 1e-7, 1e-3, log=True)

    # Run estimation
    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    # Primary objective: maximize in-sample R^2 from stage 2
    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    # define indicator for positive r2 insample stage 1
    r2_1 = float(summary.get("r2_insample_stage1", np.nan))
    positive_r2_1 = r2_1 > 0
    # define indicator for kappa t_stat greater 1.96
    kappa_tstat = float(summary.get("kappa_tstat", np.nan))
    kappa_significant = kappa_tstat > 1.96

    if not np.isfinite(r2_2):
        raise optuna.TrialPruned()

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", float(summary.get("r2_insample_stage1", np.nan)))
    trial.set_user_attr("kappa",     float(summary.get("kappa", np.nan)))
    trial.set_user_attr("kappa_tstat", float(summary.get("kappa_tstat", np.nan)))

    return r2_2 * kappa * positive_r2_1 * kappa_significant


sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study1 = optuna.create_study(direction="maximize", sampler=sampler)
study1.optimize(objective, n_trials=30, n_jobs=6, gc_after_trial=True)

[I 2026-02-18 13:22:54,424] A new study created in memory with name: no-name-2e8da4e1-06b5-4f7b-8b9f-8b1e56701f2b
[I 2026-02-18 13:23:03,302] Trial 0 finished with value: -0.0 and parameters: {'window_size': 240, 'lambda': 0.0005377777563195975}. Best is trial 0 with value: -0.0.
[I 2026-02-18 13:23:03,385] Trial 1 finished with value: 0.0 and parameters: {'window_size': 320, 'lambda': 0.0004859190288731996}. Best is trial 0 with value: -0.0.
[I 2026-02-18 13:23:03,664] Trial 3 finished with value: -0.0 and parameters: {'window_size': 310, 'lambda': 0.00026935300268198664}. Best is trial 0 with value: -0.0.
[I 2026-02-18 13:23:07,548] Trial 2 finished with value: -0.0 and parameters: {'window_size': 250, 'lambda': 3.913326114967587e-05}. Best is trial 0 with value: -0.0.
[I 2026-02-18 13:23:11,468] Trial 8 finished with value: -0.0 and parameters: {'window_size': 10, 'lambda': 4.7166274809454e-05}. Best is trial 0 with value: -0.0.
[I 2026-02-18 13:23:13,909] Trial 5 finished with valu

In [11]:
trials_df = study1.trials_dataframe(
    attrs=("number", "value", "state", "params", "user_attrs", "system_attrs")
)

# save to csv
trials_df.to_csv("optuna_study_trials.csv", index=False)

In [ ]:
# print(trials_df.columns)

Index(['number', 'value', 'state', 'params_lambda', 'params_window_size',
       'user_attrs_kappa', 'user_attrs_kappa_tstat', 'user_attrs_r2_stage1',
       'user_attrs_r2_stage2'],
      dtype='object')


In [12]:
# sort from highest R² (Optuna objective = "value") to lowest
df = trials_df.sort_values("value", ascending=False)

param_cols = [c for c in df.columns if c.startswith("params_")]

for _, row in df.iterrows():
    params = {c.replace("params_", ""): row[c] for c in param_cols}

    print(
        f"Trial {int(row['number'])}: "
        f"R²={row['value']:.6f}, "
        f"Params={params}, "
        f"kappa={row.get('user_attrs_kappa', float('nan')):.6g}, "
        f"t={row.get('user_attrs_kappa_tstat', float('nan')):.2f}, "
        f"R2_stage1={row.get('user_attrs_r2_stage1', float('nan')):.6f}, "
        f"R2_stage2={row.get('user_attrs_r2_stage2', float('nan')):.6f}"
    )


Trial 169: R²=0.000980, Params={'lambda': 5.783055944698348e-07, 'window_size': 20}, kappa=0.0819359, t=5.17, R2_stage1=0.955256, R2_stage2=0.011964
Trial 58: R²=0.000559, Params={'lambda': 1.287248939663593e-07, 'window_size': 20}, kappa=0.058678, t=4.49, R2_stage1=0.955256, R2_stage2=0.009521
Trial 85: R²=0.000525, Params={'lambda': 1.0268629243980018e-07, 'window_size': 20}, kappa=0.0568223, t=4.42, R2_stage1=0.955256, R2_stage2=0.009237
Trial 61: R²=0.000524, Params={'lambda': 1.0246087021392341e-07, 'window_size': 20}, kappa=0.0567721, t=4.41, R2_stage1=0.955256, R2_stage2=0.009226
Trial 107: R²=0.000517, Params={'lambda': 1.0020848789020976e-07, 'window_size': 20}, kappa=0.056461, t=4.39, R2_stage1=0.955256, R2_stage2=0.009151
Trial 55: R²=0.000441, Params={'lambda': 1.2692551485881178e-07, 'window_size': 30}, kappa=0.0588988, t=3.97, R2_stage1=0.973914, R2_stage2=0.007488
Trial 86: R²=0.000430, Params={'lambda': 1.039417433643531e-07, 'window_size': 30}, kappa=0.0575825, t=3.96,

In [ ]:
#save the best hyperparameters to the .txt file
